In [ ]:
# Cell 1: Install Dependencies
!pip install -q vllm cloudflared pygithub
!pip install -q PyGithub
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
import os
import re
import sys
import time
import json
import subprocess
from pathlib import Path

# ==========================================
# 1. INSTALL CLOUDFLARED (if missing)
# ==========================================
if not Path("/usr/local/bin/cloudflared").exists():
    print("📥 Installing Cloudflare CLI (cloudflared)...")
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/tmp/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/tmp/cloudflared"], check=True)
    subprocess.run(["sudo", "mv", "/tmp/cloudflared", "/usr/local/bin/cloudflared"], check=True)
    print("✅ cloudflared installed.")

# ==========================================
# 2. SET TPU ENVIRONMENT & LAUNCH vLLM
# ==========================================
print("🚀 Configuring TPU v5e-8 and starting vLLM Vision Server...")

tpu_env = os.environ.copy()
tpu_env["PJRT_DEVICE"] = "TPU"
tpu_env["VLLM_TARGET_DEVICE"] = "tpu"

vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-VL-7B-Instruct",             # Qwen Vision-Language model
    "--tensor-parallel-size", "8",                         # Spreads across all 8 TPU chips
    "--max-model-len", "65536",                            # Context window > 50K (64K tokens)
    "--limit-mm-per-prompt", '{"image": 5, "video": 0}',   # Enables vision/image inputs
    "--override-generation-config", '{"temperature": 0.0}',# Temperature set to 0 (deterministic)
    "--default-chat-template-kwargs", '{"enable_thinking": false}',  # Forces no_think mode
    "--trust-remote-code",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--gpu-memory-utilization", "0.92"
]

vllm_log = open("/tmp/vllm.log", "w")
vllm_proc = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT, env=tpu_env)

# ==========================================
# 3. LAUNCH CLOUDFLARE QUICK TUNNEL
# ==========================================
print("🌐 Opening Cloudflare Quick Tunnel...")
tunnel_log_path = "/tmp/cloudflared.log"
tunnel_log = open(tunnel_log_path, "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

public_url = None
print("⏳ Waiting for Cloudflare URL generation...")

for _ in range(30):
    time.sleep(2)
    if os.path.exists(tunnel_log_path):
        with open(tunnel_log_path, "r") as f:
            log_content = f.read()
            match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", log_content)
            if match:
                public_url = match.group(0)
                break

if not public_url:
    print("❌ Failed to capture Cloudflare URL. Check logs in /tmp/cloudflared.log")
    sys.exit(1)

# Variables setup
tunnel_url = public_url
api_base_url = f"{public_url}/v1"
GH_USER = "SCUALIMAIL"
chat_ui_url = f"https://{GH_USER.lower()}.github.io/QWEN/QWEN_Chat.html"

# Write QWEN_ENDPOINT.json locally so you can sync it to GitHub
endpoint_data = {"url": public_url}
with open("/kaggle/working/QWEN_ENDPOINT.json", "w") as endpoint_file:
    json.dump(endpoint_data, endpoint_file, indent=2)

# Save variables across notebook cells using IPython magic
%store tunnel_url
%store chat_ui_url
%store api_base_url

print("\n" + "="*65)
print(f"🎉 TPU VISION SERVER & TUNNEL READY!")
print(f"🔗 API Base URL:        {api_base_url}")
print(f"💬 Chat UI URL:         {chat_ui_url}")
print(f"🧠 Context Window:      65,536 tokens (>50K target)")
print(f"🌡️ Temperature:         0.0 (Deterministic)")
print(f"👁️ Vision Engine:       Active")
print(f"📁 Endpoint File:       Saved to /kaggle/working/QWEN_ENDPOINT.json")
print("="*65 + "\n")

# ==========================================
# 4. WAIT FOR vLLM SERVER READINESS
# ==========================================
print("⏳ Waiting for vLLM to initialize weights on TPU (this may take 2-4 mins)...")
import requests

server_ready = False
for i in range(60):
    try:
        res = requests.get("http://localhost:8000/health", timeout=3)
        if res.status_code == 200:
            server_ready = True
            print("✅ vLLM Server on TPU is LIVE and healthy!")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(5)

if not server_ready:
    print("⚠️ Warning: Health check timed out. Check /tmp/vllm.log to inspect weight compilation progress.")
    # ==========================================
# 5. AUTOMATICALLY SYNC ENDPOINT TO GITHUB
# ==========================================
import shutil

print("🔄 Automatically syncing new Cloudflare endpoint to GitHub repository...")

gh_pat = "REPLACE_ME_GH_PAT"
repo_slug = "SCUALIMAIL/QWEN"  # Your GitHub username and repo name
branch_name = "main"          # Change to "master" if your default branch is master

clone_url = f"https://{gh_pat}@github.com/{repo_slug}.git"
temp_repo_dir = "/tmp/github_repo"

try:
    # Configure git credentials
    subprocess.run(["git", "config", "--global", "user.email", "kaggle-bot@tpu.local"], check=True)
    subprocess.run(["git", "config", "--global", "user.name", "Kaggle TPU Automation Bot"], check=True)

    # Clone repository cleanly
    if os.path.exists(temp_repo_dir):
        shutil.rmtree(temp_repo_dir)
        
    subprocess.run(["git", "clone", "--depth", "1", "--branch", branch_name, clone_url, temp_repo_dir], check=True)

    # Copy generated endpoint file into the repo's QWEN subfolder
    target_json_path = os.path.join(temp_repo_dir, "QWEN", "QWEN_ENDPOINT.json")
    shutil.copy("/kaggle/working/QWEN_ENDPOINT.json", target_json_path)

    # Commit and push changes
    subprocess.run(["git", "-C", temp_repo_dir, "add", "QWEN/QWEN_ENDPOINT.json"], check=True)
    
    # Check if there are actual changes to commit
    status_res = subprocess.run(["git", "-C", temp_repo_dir, "status", "--porcelain"], capture_output=True, text=True, check=True)
    if status_res.stdout.strip():
        subprocess.run(["git", "-C", temp_repo_dir, "commit", "-m", "Auto-update QWEN_ENDPOINT.json from Kaggle TPU [skip ci]"], check=True)
        subprocess.run(["git", "-C", temp_repo_dir, "push", clone_url, branch_name], check=True)
        print("✅ Successfully pushed updated QWEN_ENDPOINT.json back to GitHub!")
    else:
        print("ℹ️ Endpoint URL hasn't changed; no commit needed.")

except Exception as e:
    print(f"⚠️ Warning: Automatic GitHub sync failed: {e}")

In [ ]:
# Cell 3: Update GitHub JSON Endpoint
import json
import time
from kaggle_secrets import UserSecretsClient
from github import Github

# Fetch GitHub PAT from Kaggle Secrets
secrets = UserSecretsClient()
GH_PAT = secrets.get_secret("GH_PAT")

# Configuration
REPO_NAME = "SCUALIMAIL/SCUALIMAIL.github.io"
FILE_PATH = "QWEN/QWEN_ENDPOINT.json"
BRANCH = "main"

# Retrieve tunnel variables from Cell 2
%store -r tunnel_url
%store -r chat_ui_url
%store -r api_base_url

if not tunnel_url:
    raise ValueError("Tunnel URL missing. Please run Cell 2 first.")

# Connect to GitHub
g = Github(GH_PAT)
repo = g.get_repo(REPO_NAME)

payload = {
    "status": "online",
    "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "tunnel_url": tunnel_url,
    "chat_ui_url": chat_ui_url,
    "api_base_url": api_base_url,
    "temperature": 0,
    "max_context_window": 65536
}

json_content = json.dumps(payload, indent=2)

try:
    # Get existing file to retrieve SHA
    contents = repo.get_contents(FILE_PATH, ref=BRANCH)
    repo.update_file(
        path=FILE_PATH,
        message="Update Qwen dynamic endpoints [Kaggle Auto-sync]",
        content=json_content,
        sha=contents.sha,
        branch=BRANCH
    )
    print(f"✅ Successfully updated {FILE_PATH} on branch '{BRANCH}'.")
except Exception as e:
    # Create file if it does not exist
    repo.create_file(
        path=FILE_PATH,
        message="Create Qwen dynamic endpoints [Kaggle Auto-sync]",
        content=json_content,
        branch=BRANCH
    )
    print(f"✅ Created new file {FILE_PATH} on branch '{BRANCH}'.")

print(f"🔗 GitHub Landing Page URL: https://scualimail.github.io/QWEN/QWEN_Chat.html")